In [1]:
import numpy as np
import scipy.sparse as sp

from math import sqrt

import igl
import pyvista as pv
pv.set_jupyter_backend('trame')

In [2]:
def to_pyvista_mesh(V, F):
    return pv.UnstructuredGrid({pv.CellType.TRIANGLE: F}, V)

In [68]:
v, f = igl.read_triangle_mesh("bunny.off")

In [69]:
mesh = to_pyvista_mesh(v, f)
mesh.plot(show_edges=True)

Widget(value='<iframe src="http://localhost:53756/index.html?ui=P_0x31ef69ca0_9&reconnect=auto" class="pyvista…

In [70]:
# each idx corresponds to a face, each element is a component, highest component count = biggest component\
# Filter out noise components
(num_components, component_ids) = igl.facet_components(f)
unique, counts = np.unique(component_ids, return_counts=True)
count_per_component = dict(zip(unique, counts))

comp_vals = list(count_per_component.values())
largest_comp_idx = comp_vals.index(max(comp_vals))
filtered_f_idx = [i for i, comp in enumerate(component_ids) if comp == largest_comp_idx]

filtered_f = f[filtered_f_idx, :]

to_pyvista_mesh(v, filtered_f).plot(show_edges=True)

Widget(value='<iframe src="http://localhost:53756/index.html?ui=P_0x33a0d7e80_10&reconnect=auto" class="pyvist…

In [71]:
vf, ni = igl.vertex_triangle_adjacency(f, len(v))

# TODO: Check double areas, should be 1/2 or no?
face_areas = 1/2 * igl.doublearea(v, f)
angles = igl.internal_angles(v, f)

A_flat = []
L_c = np.zeros((len(v), len(v)))
for i in range(len(v)):
    adj_faces = []
    # Find all incident faces for current vertex
    for j in range(0, ni[i+1] - ni[i]):
        cur_face = vf[ni[i] + j]
        adj_faces.append(cur_face)
        
        # Find a_ij and b_ij for i
        assoc_tri_verts = f[cur_face]
        angle_idx = np.where(assoc_tri_verts != i)[0]

        # L_c at [i, adj_vert a or b] = angle a_ij, b_ij
        L_c[i, assoc_tri_verts[angle_idx[0]]] = 1 / np.tan(angles[i, angle_idx[0]]) # cot(angle)
        L_c[i, assoc_tri_verts[angle_idx[1]]] = 1 / np.tan(angles[i, angle_idx[1]])
        
    A_i = 1/3 * sum(face_areas[adj_faces])
    
    A_flat.append(A_i)
assert(len(A_flat) == len(v))
A = np.diag(A_flat)

In [72]:
t = igl.avg_edge_length(v, f) ** 2 # Paper gives t = h^2 as a good heuristic
dirac_delta = np.zeros(len(v))
dirac_delta[0] = 1 # Initialize starting point to vertex zero

# TODO: Convert to sparse mat, takes a long time to solve for larger meshes
u = np.linalg.solve(A - t * L_c, dirac_delta)

assert(len(u) == len(v))

In [73]:
X = np.zeros((len(f), 3))
face_normals = igl.per_face_normals(v, f)

for face_idx, face_verts in enumerate(f):
    cur_face_normal = face_normals[face_idx]
    
    # Summation
    sigma_i = 0
    for vert in face_verts:
        u_i = u[vert]
        opposite_edge = np.where(face_verts != vert)[0]
        e_i = v[opposite_edge[1]] - v[opposite_edge[0]]
        
        # u_i * (N x e_i)
        sigma_i = u_i * np.cross(cur_face_normal, e_i)
    grad_u = 1/(2 * face_areas[face_idx]) * sigma_i
    X[face_idx] = - grad_u / np.linalg.norm(grad_u)

In [74]:
div_X = np.zeros(len(v))
for i in range(len(v)):
    adj_faces = []

    cur_sum = 0
    # Find all incident faces for current vertex
    for j in range(0, ni[i+1] - ni[i]):
        X_j = X[j]
        
        cur_face = vf[ni[i] + j]

        # Find a_ij and b_ij for i
        assoc_tri_verts = f[cur_face]
        angle_idx = np.where(assoc_tri_verts != i)[0]
        
        # theta_1 and theta_2 are opposing angles of i 
        theta_1 = angles[i, angle_idx[0]]
        theta_2 = angles[i, angle_idx[1]]

        # e_1 and e_2 are edge vectors containing i
        e_1 = v[assoc_tri_verts[angle_idx[1]]] - v[i]
        e_2 = v[assoc_tri_verts[angle_idx[0]]] - v[i]
            
        cur_sum += (1 / np.tan(theta_1)) * (np.dot(e_1, X_j)) + (1 / np.tan(theta_2)) * (np.dot(e_2, X_j))
            
    div_X[i] = 1/2 * cur_sum
    

In [83]:
b = div_X
phi = np.linalg.solve(L_c, b)
phi = phi - phi.min() # adjust

assert(len(phi) == len(v))


In [84]:
pl = pv.Plotter()
pl.add_points(mesh.points[0], color='red', point_size=20) # Add visual to starting point
pl.add_mesh(mesh, scalars=phi, show_edges=True)
pl.show()

Widget(value='<iframe src="http://localhost:53756/index.html?ui=P_0x3a9e4b700_19&reconnect=auto" class="pyvist…